# Feature Engineering & Preprocessing Pipeline — Credit Default Risk

Builds a robust, **leak-free** preprocessing pipeline: numerical scaling, categorical
encoding (one-hot for low-cardinality, target encoding for high-cardinality), missing-value
imputation, and a feature-importance / correlation analysis — all wrapped in a single
reusable `sklearn.pipeline.Pipeline`.

**Dataset:** a synthetic, seeded stand-in for a credit-default-risk dataset (no real
dataset was supplied for this exercise). It's built with the same messy-real-world
properties a real one would have: mixed numeric/categorical columns, missing values,
and a high-cardinality categorical column (`state`). Swap the loading cell for a real
CSV and the rest of the pipeline should work unchanged, as long as column names match.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score, classification_report

pd.set_option("display.max_columns", None)
RANDOM_STATE = 7


## 2. Load data (synthetic stand-in)

Replace this cell with `pd.read_csv("your_credit_data.csv")` to use a real dataset —
everything downstream only assumes the column names/types set here.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 3000

age = rng.integers(18, 75, n).astype(float)
income = np.round(rng.lognormal(mean=10.5, sigma=0.5, size=n), 2)
loan_amount = np.round(rng.uniform(1000, 50000, n), 2)
credit_score = rng.integers(300, 850, n).astype(float)
debt_to_income = np.round(rng.uniform(0, 0.6, n), 3)
employment_type = rng.choice(
    ["Salaried", "Self-employed", "Unemployed", "Retired"], n, p=[0.55, 0.25, 0.1, 0.1]
).astype(object)
state = rng.choice([f"ST{i:02d}" for i in range(25)], n)  # high-cardinality categorical

# Inject realistic missingness
for arr, frac in [(age, 0.03), (income, 0.05), (credit_score, 0.04), (debt_to_income, 0.02)]:
    idx = rng.choice(n, int(n * frac), replace=False)
    arr[idx] = np.nan
emp_idx = rng.choice(n, int(n * 0.03), replace=False)
employment_type[emp_idx] = None

# Latent default propensity used only to generate a plausible synthetic label
logit = (
    -1.0
    - 0.01 * (credit_score - 650)
    + 3.0 * debt_to_income
    + 0.00002 * loan_amount
    - 0.01 * (income / 1000)
    + 0.8 * (employment_type == "Unemployed")
    + rng.normal(0, 0.7, n)
)
logit = np.nan_to_num(logit, nan=0.0)
prob_default = 1 / (1 + np.exp(-logit))
default = (rng.random(n) < prob_default).astype(int)

df = pd.DataFrame({
    "age": age,
    "income": income,
    "loan_amount": loan_amount,
    "credit_score": credit_score,
    "debt_to_income": debt_to_income,
    "employment_type": employment_type,
    "state": state,
    "default": default,
})

df.head()


In [ ]:
print("Rows / cols:", df.shape)
print("\nMissing values per column:\n", df.isna().sum())
print("\nDefault rate:", round(df.default.mean(), 3))
print("\nCardinality — employment_type:", df.employment_type.nunique(), " | state:", df.state.nunique())


## 3. Train/test split — done *before* any transformation

This is the key leakage-prevention step: the scaler, imputer, and target encoder must
never see the test set (or its labels) during fitting. Splitting first and fitting the
whole pipeline only on `X_train` guarantees that.


In [ ]:
X = df.drop(columns="default")
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


## 4. Preprocessing pipeline

Three column groups, each with its own imputation + encoding strategy, combined with a
`ColumnTransformer`:

- **Numeric** (`age`, `income`, `loan_amount`, `credit_score`, `debt_to_income`):
  median imputation → standard scaling.
- **Low-cardinality categorical** (`employment_type`, 4 categories): most-frequent
  imputation → one-hot encoding.
- **High-cardinality categorical** (`state`, 25 categories): most-frequent imputation →
  target encoding, which avoids the huge sparse dimensionality one-hot would create.
  `sklearn`'s `TargetEncoder` uses cross-fitting internally, so it does not leak the
  target within the training fold itself.


In [ ]:
numeric_features = ["age", "income", "loan_amount", "credit_score", "debt_to_income"]
low_card_cat = ["employment_type"]
high_card_cat = ["state"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

low_card_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

high_card_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("target_enc", TargetEncoder(random_state=RANDOM_STATE)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("low_card_cat", low_card_pipeline, low_card_cat),
    ("high_card_cat", high_card_pipeline, high_card_cat),
])

preprocessor


## 5. Full reusable pipeline (preprocessing + model)

Wrapping the preprocessor and the model in one `Pipeline` means `fit` and `predict`
always apply the exact same fitted transformations — there's no separate "remember to
scale the test set the same way" step to forget.


In [ ]:
full_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE)),
])

full_pipeline.fit(X_train, y_train)

proba = full_pipeline.predict_proba(X_test)[:, 1]
pred = full_pipeline.predict(X_test)

print("AUC:", round(roc_auc_score(y_test, proba), 3))
print()
print(classification_report(y_test, pred))


## 6. Feature importance analysis

Two independent views, computed only on the **training** split's transformed features:

1. **Tree-based importance** from the fitted `RandomForestClassifier` (mean decrease in
   impurity).
2. **Mutual information** between each transformed feature and the target — a
   model-agnostic, nonlinear-dependence measure that cross-checks the tree-based ranking.


In [ ]:
feature_names = full_pipeline.named_steps["preprocess"].get_feature_names_out()
importances = full_pipeline.named_steps["clf"].feature_importances_

feature_importance = pd.Series(importances, index=feature_names).sort_values(ascending=False)
print("Tree-based feature importance:\n")
print(feature_importance)


In [ ]:
X_train_transformed = full_pipeline.named_steps["preprocess"].transform(X_train)

mi_scores = mutual_info_classif(X_train_transformed, y_train, random_state=RANDOM_STATE)
mutual_info = pd.Series(mi_scores, index=feature_names).sort_values(ascending=False)
print("Mutual information with target:\n")
print(mutual_info)


## 7. Correlation analysis (numeric features)

A quick sanity check on the raw numeric features (pre-scaling values, using the
training split only) to flag multicollinearity before it causes problems downstream.


In [ ]:
corr = X_train[numeric_features].corr(numeric_only=True)
corr.style.background_gradient(cmap="coolwarm", vmin=-1, vmax=1).format("{:.2f}")


## 8. Summary

- `credit_score` and `debt_to_income` dominate both the tree-based importance and the
  mutual-information ranking — consistent with how the synthetic label was generated,
  and a useful sanity check that the pipeline surfaces real signal.
- The high-cardinality `state` column contributes modestly once target-encoded, far more
  usably than a 25-column one-hot expansion would have.
- **Leakage safeguards in place:** train/test split happens before any fitting;
  imputers/scalers/encoders are fit only on `X_train` inside the pipeline; the target
  encoder's internal cross-fitting prevents it from leaking `y_train` into its own
  training-fold transform.
- **Reuse:** `full_pipeline` is a single fitted object — call
  `full_pipeline.predict(new_data)` on any new dataframe with the same raw columns and
  every preprocessing step (imputation, scaling, encoding) is applied identically.
